# Trace GaiaXPy results with metadata

GaiaXPy records metadata for generated results, which include package version, configuration file path and version, options applied, etc. This tutorial uses standardised photometry to find the package version, the configuration and its paper reference.

**Contents**

- [Metadata in memory](#gaiaxpy-metadata-in-memory)
- [Configuration and reference](#gaiaxpy-configuration-and-reference)
- [Standardisation rules](#gaiaxpy-standardisation-rules)
- [Metadata in saved files](#gaiaxpy-metadata-in-saved-files)
- [Metadata in configuration files](#gaiaxpy-metadata-in-config-files)

<a id="gaiaxpy-metadata-in-memory" name="gaiaxpy-metadata-in-memory"></a>

## Metadata in memory

This example uses DR3 continuous XP mean spectra and a DR3 standardised photometric system. The returned DataFrame stores processing provenance in `attrs["gaiaxpy_metadata"]`.


In [ ]:
from gaiaxpy import PhotometricSystem, generate

spectra_input = "path/to/dr3_xp_mean_spectra.ecsv"
system = PhotometricSystem.DR3_JKC_Std
photometry = generate(spectra_input, photometric_system=system, save_file=False)
metadata = photometry.attrs["gaiaxpy_metadata"]
metadata.keys()  # schema_version refers to the metadata schema

<a id="gaiaxpy-configuration-and-reference" name="gaiaxpy-configuration-and-reference"></a>

## Configuration and reference

In the particular case of a standardised photometric system, the `standardisation` entries identify the files used. The matching entries in `configuration_files` contain the configuration version and any recorded reference.

In [ ]:
files = {entry["file"] for entry in metadata["standardisation"]}
for entry in metadata["configuration_files"]:
    if entry["file"] in files:
        details = entry["metadata"]
        print(entry["file"], details.get("version"))
        if "reference" in details:
            print(details["reference"])

The metadata contains, among others, the path to the photometric filter file used, the filter version (`1`) and the corresponding reference paper that contains the relevant information about the generation of synthetic photometry.

<a id="gaiaxpy-standardisation-rules" name="gaiaxpy-standardisation-rules"></a>

## Standardisation and provenance

DR3 standardised photometric systems apply configured offsets. You can inspect the selected system's offsets directly. The output metadata records the configuration involved in the run; the corresponding configuration metadata provides its reference where available.


In [ ]:
system.get_offsets()  # Configured flux-bias offsets, in passband order
system.get_bands()

<a id="gaiaxpy-metadata-in-saved-files" name="gaiaxpy-metadata-in-saved-files"></a>

## Metadata in saved files

This example saves the same photometry calculation as CSV and reads its metadata back. CSV uses a companion file ending in `_metadata.csv`. For other formats, the metadata is contained in the output file itself.

In [ ]:
from gaiaxpy.output import read_output_metadata

saved = generate(
    spectra_input, photometric_system=system, save_file=True,
    output_path="results", output_file="synthetic_photometry", output_format="csv"
)
saved_metadata = read_output_metadata("results/synthetic_photometry.csv")
saved_metadata["standardisation"]

<a id="gaiaxpy-metadata-in-config-files" name="gaiaxpy-metadata-in-config-files"></a>

## Metadata in configuration files

GaiaXPy includes configuration files with the package. The same lookup works with a pip installation or a source checkout. The variable `config_path` points to the configuration directory of the GaiaXPy package in use.

The example reads the metadata from a bundled DR3 calibration file.

In [ ]:
from gaiaxpy.config.paths import config_path
from gaiaxpy.core.config_metadata import read_config_metadata

config_file = config_path / "calibrator_config" / "dr3" / "basic" / "bp_dr3_v375wi_bases.npz"
read_config_metadata(config_file)